---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-40: Agentic AI and Tool Calling - II</h1>

# Learning agenda of this notebook
1. Example: Calling a Custom Function (Multiply) using OpenAI's `Responses` API using Gorq Hosted Model
    - Step 1: Define the Function and Tool Schema
    - Step 2: Ask the Model (Initial Request)
    - Step 3: Make the Tool call (if Required) and Send the tool result to Model
    - Step 4: Display the final result
2. Example: Calling a Custom Function (Multiply) using OpenAI's `Chat Completion` API using Groq Hosted Model
3. Example: Calling a Custom Function (Get Current Date/Time) Using OpenAI's `Responses` API using Groq Hosted Model
4. Example: Calling a Custom Function (Get Weather via API) Using OpenAI's `Responses` API using Groq Hosted Model
5. Example: Multiply Tool Call with a **Conversational Agent (Gradio app)** with `Chat Completion` API using OpenAI's gpt-4o-mini
6. Example: Get Weather Tool Call with a **Conversational Agent (Gradio app)** with `Chat Completion` API using OpenAI's gpt-4o-mini
7. Example: Executing Shell commands on Local Box with a **Conversational Agent (Gradio app)** with Chat Completion API using OpenAI's gpt-4o-mini

# <span style='background :lightgreen' >Recap: Workflow of a Tool Call</span>

<div style="text-align:center;">
    <img src="../images/tool-call-workflow.svg"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >1. Example: Calling a Custom Function (Multiply) using OpenAI's `Responses` API using Groq Hosted Model</span>

## Step 1: Define the Tool and Tool Schema (Tool Specification)
- You define the actual Python function that executes when the model calls your tool. Specify proper type hints and docstring so the LLM understands the expected inputs and return type.
- Tool Schema (JSON) describes your tool to the model - what it does, what parameters it accepts, and when to use it. This is what the model "sees" and uses to decide whether to call your tool. 
- Create a flattened JSON schema for the Responses API with four essential elements:
    - **type:** Specifies that this tool represents a callable function. In this code → "function". For user-provided tools, type="function" is the only supported option.
    - **name:** A short descriptifve function identifier. In this code →  "multiply`"
    - **description:** A short descriptifve function identifier/ In this code → "Multiply two numbers"
    - **parameters:** A JSON Schema object describing the input fields, their types, and any required arguments.
- This schema tells the LLM what functions are available and how to call them
- Key difference from Chat Completions: No nested "function" wrapper object

In [1]:
# ================================================================================================================================
# Define the Python function that will execute on local machine
# Do add type hinting and doc string as it will help LLM to know what type of data is in the input and what is the type of return value
# =================================================================================================================================
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

# ===============================================================================================================================================================================================
# Define the tool schema in JSON. The following schema tells the model: "You may call a function named multiply, which requires a JSON object with two numeric fields (a and b), both mandatory."
# ===============================================================================================================================================================================================
tools = [{
    "type": "function",           # "type" defines the kind of tool being declared. For the Responses API, valid top-level type values include: function, computer, code_interpreter. 
    "name": "multiply",           # Public unique identifier of the tool/function. The LLM mentions this name when calling the tool (inside tool_calls). Must match the actual function name you will execute locally.
    "description": "Multiply two numbers", # "description" is human readable explanation of what the tool does that helps both developers and LLMs understand: when to call tool, what the tool achieves etc
    "parameters": {               # "parameters" field describes what arguments the model must provide when calling your tool.
        "type": "object",         # type:object insider parameters describes that tool expects a JSON object (key-value pairs). Other rarely used types can be string, number, array, boolean or null.
        "properties": {           # "properties" defines the allowed and expected fields inside the object. Each property has its own JSON schema (Here we have two parameters "a" and "b")
            "a": {"type": "number"},
            "b": {"type": "number"}
        },
        "required": ["a", "b"]    # "required" specifies which parameters are mendatory. Missing a required field triggers immediate validation failure
    }
}]
print("=== STEP 1: Tool Defined (Responses API) ===")
print(f"Available tool: {tools[0]['name']}") 

=== STEP 1: Tool Defined (Responses API) ===
Available tool: multiply


## Step 2: Ask the Model (Initial Request)
- Now you send your initial request to the model, specifying the `tools` parameter as the tool schema you created and the `tool_choice` role as auto
- Send your user query using `client.responses.create()` with:
    - input=user_query (string format, not messages array)
    - tools=tools (your function definitions)
    - tool_choice="auto" (let model decide when to call functions)
- The model analyzes the request and determines if it needs to call a function
- Parse the response by iterating through response.output looking for items with type="function_call"
- Extract function name from item.name and arguments from json.loads(item.arguments)
- Key difference: Look for "function_call" type, not "tool_call"

In [3]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.

print("\n=== STEP 2: Initial Request to the Model ===")
#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What is 6 times 7?"                                # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
    input=user_query,  
    tools=tools,
    tool_choice="auto" # auto, required, none
)

# =============================================================================================================
# Iterate through the response.output and check `item.type=="function_call"` to check if a tool call is required
# =============================================================================================================
tool_call = None
args = None
for item in response.output:
    if item.type == "function_call":                # Model requested function # Check for 'function_call' type (not 'tool_call')
        tool_call = item
        print(f"Tool call: {tool_call}")
        print(f"Tool requested: {tool_call.name}")
        args = json.loads(tool_call.arguments)      # Parse the arguments from the function call
        print(f"Arguments parsed: {args}")
        break


=== STEP 2: Initial Request to the Model ===
Tool call: ResponseFunctionToolCall(arguments='{"a":6,"b":7}', call_id='gqa69gfdd', name='multiply', type='function_call', id='gqa69gfdd', namespace=None, status='completed')
Tool requested: multiply
Arguments parsed: {'a': 6, 'b': 7}


## Step 3: Make the Tool call (if Required) and Send the tool result to Model
- If tool call is made, you need to execute the function with appropriate arguments and save the result in say `tool_result` variable 
- For `client.chat.completions.create()` call the Model again with tool results included int the structured role messages (user, assistant, tool)
- For `client.responses.create()` call the Model again with tool results in appropriate format required by the Responses API
>- If tool call is NOT made simply display the initial response

In [4]:
print("\n=== STEP 3: Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response
# Check if the model wants to call a tool
if tool_call:
    print("Executing the function")
    tool_result = multiply(args["a"], args["b"])
    # Make second API call with the tool result
    print("Making the second call, and sending result of tool call for formatting")
    # ================================================================
    # Send tool result back to model - RESPONSES API FORMAT
    # ================================================================
    context_message = f"""
    The user asked: "{user_query}"
    I called the multiply function with a={args['a']} and b={args['b']} and got result: {tool_result}
    Please provide a natural response to the user.
    """
    
    final_response = groq_client.responses.create(
            model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
            input=context_message
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response


=== STEP 3: Check if the Tool call is Required and make second call if needed ===
Executing the function
Making the second call, and sending result of tool call for formatting


## Step 4: Display the final result

In [5]:
print("\n=== STEP 4: Final Result ===")
if tool_call:
    print("Result from second API call (after tool execution):")
else:
    print("Result from first API call (direct response):")
print(final_response.output_text)


=== STEP 4: Final Result ===
Result from second API call (after tool execution):
The result of 6 times 7 is 42.


# <span style='background :lightgreen' >2. Example: Calling a Custom Function (Multiply) using OpenAI's `Chat Completion` API  using Groq Hosted Model</span>

## Step 1: Define the Function and Tool Schema

In [ ]:
# ================================================================================================================================
# Define the Python function that will execute on local machine
# Do add type hinting and doc string as it will help LLM to know what type of data is in the input and what is the type of return value
# =================================================================================================================================
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

# ================================================================
# Define tool schema (inside 'function' object)
# ================================================================
tools = [{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["a", "b"]
        }
    }
}]

print("=== STEP 1: Tool Defined (Chat Completion API) ===")
print(f"Available tool: {tools[0]['function']['name']}")

## Step 2: Ask the Model (Initial Request)

In [ ]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from groq import Groq                        # Official Groq Python client for interacting with Groq-hosted LLMs via their API.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.


print("\n=== STEP 2: Initial Request to the Model ===")
#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What is 6 times 7?"                                # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
    messages=[{"role": "user", "content": user_query}],
    tools=tools,
    tool_choice="auto"
)
# Check if the model wants to call a tool
if response.choices[0].message.tool_calls:
    print("Tool call detected.")
else:
    print("No tool call needed.")

## Step 3: Make the Tool call (if Required) and Send the tool result to Model

In [ ]:
print("\n=== STEP 3: Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response

# Check if the model wants to call a tool
if response.choices[0].message.tool_calls:
    print("Tool call detected!")
    
    # Extract tool call details
    tool_call = response.choices[0].message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    # Execute the Python function (in this case, multiply)
    if function_name == "multiply":
        print("Executing the function")
        tool_result = multiply(function_args["a"], function_args["b"])
        
        # Make second API call with the tool result
        print("Making the second call, and sending result of tool call for formatting")
# ===============================================================
# Generate the messages with user query, assistant's tool call and tool's result
# ===============================================================
    input = [
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": None,  # Assistant gave no direct text, only tool call
                                            "tool_calls": [{
                                                            "id": tool_call.id,
                                                            "type": "function",
                                                            "function": {"name": tool_call.function.name, "arguments": tool_call.function.arguments}
                                                            }]
        },
        {"role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)}
    ]

    final_response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
            messages=input,
            tools=tools,
            tool_choice="auto"
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response

## Step 4: Display the final result

In [ ]:
print("\n=== STEP 4: Final Result ===")
if response.choices[0].message.tool_calls:
    print("Result from second API call (after tool execution):")
else:
    print("Result from first API call (direct response):")

print(final_response.choices[0].message.content)

# <span style='background :lightgreen' >3. Example: Calling a Custom Function (Get Current Date/Time) Using OpenAI's `Responses` API  using Groq Hosted Model</span>

In [6]:
import os                                   # Provides access to environment variables and operating system utilities.
from datetime import datetime
import pytz
import rich                                 # Renders richly formatted output (colors, tables, tracebacks) in the terminal.
import json                                 # Encodes and decodes data in JSON format.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not

openai_api_key = os.getenv('OPENAI_API_KEY') # The os.getenv() method reads the environment variables and returns the value associated with 'OPENAI_API_KEY', or None if it is not set.
groq_api_key = os.getenv("GROQ_API_KEY")

openai_client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)  # This client is your gateway to native OpenAI models like gpt-4o, gpt-4o-mini, etc.
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.



# ================================================================================================================================
# Define the Python function that will execute on local machine
# Do add type hinting and doc string as it will help LLM to know what type of data is in the input and what is the type of return value
# =================================================================================================================================
def get_current_time(timezone: str) -> str:
    """
    Get the current date and time for a specific timezone.
    Args:
        timezone (str): IANA timezone identifier (e.g., 'Asia/Karachi')
    Returns:
        str: Formatted current time string
    """
    tz = pytz.timezone(timezone)     # Create timezone object
    now = datetime.now(tz)       # Get current time in that timezone
    return now.strftime("%Y-%m-%d %H:%M:%S %Z") # Format and return the time

# =================================================================================================================================================================================================================================
# Define the tool schema in JSON. The following schema tells the model: "You may call a function named get_current_time by providing a JSON object that contains one required string field, timezone, representing an IANA timezone.
# =================================================================================================================================================================================================================================
tools = [{
    "name": "get_current_time",  # Public unique identifier of the tool/function. The LLM mentions this name when calling the tool (inside tool_calls). Must match the actual function name you will execute locally.
    "type": "function",          # "type" defines the kind of tool being declared. For the Responses API, valid top-level type values include: function, computer, code_interpreter. 
    "description": "Get current date and time for a given timezone",  # "description" is human readable explanation of what the tool does that helps both developers and LLMs understand: when to call tool, what the tool achieves etc
    "parameters": {              # "parameters" field describes what arguments the model must provide when calling your tool.
        "type": "object",        # type:object insider parameters describes that tool expects a JSON object (key-value pairs). Other rarely used types can be string, number, array, boolean or null.
        "properties": {          # "properties" defines the allowed and expected fields inside the object. Each property has its own JSON schema. (Here we have one parameter named "timezone")
            "timezone": {
                "type": "string",
                "description": "IANA timezone name, e.g., Asia/Karachi, America/New_York"
            }
        },
        "required": ["timezone"] # "required" specifies which parameters are mendatory. Missing a required field triggers immediate validation failure
    }
}]





#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What's the current time in Karachi?"               # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
    input=user_query,  
    tools=tools,
    tool_choice="auto" 
)
# ================================================================
# Parse tool calls from response.output, as this print(response.choices[0].message.tool_calls) will not work
# ================================================================
tool_call = None
args = None
for item in response.output:
    if item.type == "function_call":                # Model requested function # Check for 'function_call' type (not 'tool_call')
        tool_call = item
        print(f"Tool call: {tool_call}")
        args = json.loads(tool_call.arguments)      # Parse the arguments from the function call
        break




# Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response
# Check if the model wants to call a tool
if tool_call:
    tool_result = get_current_time(args["timezone"])
    # Make second API call with the tool result
    # ================================================================
    # Send tool result back to model - RESPONSES API FORMAT
    # ================================================================
    context_message = f"""
The user asked: "{user_query}"
I called the get_current_time function with timezone='{args['timezone']}' and received this result:
{tool_result}
Please provide a natural response to the user based on this information.
"""
    
    final_response = groq_client.responses.create(
            model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
            input=context_message
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response


# Final Result
print(final_response.output_text)

Tool call: ResponseFunctionToolCall(arguments='{"timezone":"Asia/Karachi"}', call_id='hzhz5gxj8', name='get_current_time', type='function_call', id='hzhz5gxj8', namespace=None, status='completed')
The current time in Karachi is 10:54 AM PKT.


# <span style='background :lightgreen' >4. Example: Calling a Custom Function (Get Weather via API) Using OpenAI's `Responses` API  using Groq Hosted Model</span>

## Accessing Weather API from Python using their API key

In [7]:
# A simple run of calling the weatherapi.com
import requests
import os
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True) 
weather_api_key = os.getenv('WEATHER_API_KEY')
city='Lahore'

url = f'http://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={city}'
response = requests.get(url)
response.raise_for_status()  # Raise an exception for bad status codes
data = response.json()
print(f"Country: {data['location']['country']}")
print(f"City: {data['location']['name']}")
print(f"Latitude: {data['location']['lat']}")
print(f"Longitude: {data['location']['lon']}")
print(f"Temperature: {data['current']['temp_c']}°C ({data['current']['temp_f']}°F)")
print(f"Condition: {data['current']['condition']['text']}")
print(f"Humidity: {data['current']['humidity']}%")
print(f"Wind: {data['current']['wind_kph']} km/h")
print(f"Feels like: {data['current']['feelslike_c']}°C")
print(f"Last updated: {data['current']['last_updated']}")

Country: Pakistan
City: Lahore
Latitude: 31.5497
Longitude: 74.3436
Temperature: 32.2°C (90.0°F)
Condition: Sunny
Humidity: 32%
Wind: 14.0 km/h
Feels like: 30.0°C
Last updated: 2026-05-07 10:45


In [8]:
import os                                   # Provides access to environment variables and operating system utilities.
from datetime import datetime
import requests
import pytz
import rich                                 # Renders richly formatted output (colors, tables, tracebacks) in the terminal.
import json                                 # Encodes and decodes data in JSON format.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) 
weather_api_key = os.getenv('WEATHER_API_KEY')
groq_api_key = os.getenv("GROQ_API_KEY")
#openai_client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)  # This client is your gateway to native OpenAI models like gpt-4o, gpt-4o-mini, etc.
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.


# ===================================================================
# Define the function that calls the weather API
# The Tool/Function `get_city_weather("lahore")` will call the weather condition from http://weatherapi.com
# ===================================================================
def get_city_weather(city_name):
    """
    Function to retrieve weather information for a given city.
    Fetches data from WeatherAPI and returns formatted weather information.
    Args:
        city_name (str): The city name to get weather for
    Returns:
        str: Weather information or error message if city not found
    """
    print(f"Tool get_city_weather called for {city_name}")
    city = city_name.strip() # Strip whitespace for consistent processing
    url = f'http://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={city}' # Build API URL
    try:
        response = requests.get(url) # Make API request
        response.raise_for_status()  # Raise error for bad responses (4xx, 5xx)
        weather_data = response.json()
        
        # Extract relevant weather information
        current = weather_data['current']
        location = weather_data['location']
        weather_info = f"Current weather in {location['name']} (Lat: {location['lat']}, Lon: {location['lon']}) as of {location['localtime']}: {current['temp_c']}°C, {current['condition']['text']}"
        return weather_info
        
    except requests.RequestException as e:
        print(f"❌ Error fetching weather data: {e}")
        return "Weather information not available for this city"
    except KeyError as e:
        print(f"Error parsing weather data: {e}")
        return "Weather information not available for this city"

# =================================================================================================================================================================================================================================
# Define the tool schema in JSON. The following schema tells the model: "You may call a function named get_city_weather by providing a JSON object that contains one required string field, city_name.
# =================================================================================================================================================================================================================================
tools = [{
    "name": "get_city_weather",  
    "type": "function",         
    "description": "Get the current weather information for a city. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
    "parameters": {              # "parameters" field describes what arguments the model must provide when calling your tool.
        "type": "object",        # type:object insider parameters describes that tool expects a JSON object (key-value pairs). Other rarely used types can be string, number, array, boolean or null.
        "properties": {          # "properties" defines the allowed and expected fields inside the object. Each property has its own JSON schema. (Here we have one parameter named "timezone")
            "city_name": {
                "type": "string",
                "description": "The city that the customer wants to know the weather for, e.g., Lahore, Karachi"
            }
        },
        "required": ["city_name"],
        "additionalProperties": False
    }
}]

#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What's the current weather in Lahore?"               # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
    input=user_query,  
    tools=tools,
    tool_choice="auto" 
)
# ================================================================
# Parse tool calls from response.output, as this print(response.choices[0].message.tool_calls) will not work
# ================================================================
tool_call = None
args = None
for item in response.output:
    if item.type == "function_call":                # Model requested function # Check for 'function_call' type (not 'tool_call')
        tool_call = item
        args = json.loads(tool_call.arguments)      # Parse the arguments from the function call
        break

# Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response
# Check if the model wants to call a tool
if tool_call:
    tool_result = get_city_weather(args["city_name"])
    # Make second API call with the tool result
    # ================================================================
    # Send tool result back to model - RESPONSES API FORMAT
    # ================================================================
    context_message = f"""
The user asked: "{user_query}"
I called the get_city_weather function with timezone='{args['city_name']}' and received this result:
{tool_result}
Please provide a natural response to the user based on this information.
"""
    
    final_response = groq_client.responses.create(
            model="llama-3.3-70b-versatile",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
            input=context_message
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response

# Final Result
print(final_response.output_text)

Tool get_city_weather called for Lahore
The current weather in Lahore is 32.2°C and sunny as of 10:54 AM. It's quite warm outside, so you might want to stay hydrated and plan accordingly if you have any outdoor activities planned.


# <span style='background :lightgreen' >5. Multiply Tool Call with a **Conversational Agent (Gradio app)** with Chat Completion API using OpenAI's gpt-4o-mini</span>

In [9]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

# ===============================================================
# Load environment variables and initialize OpenAI client
# ===============================================================
load_dotenv("../keys/.env", override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ===============================================================
# Define the multiply tool schema
# ===============================================================
tools = [{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers together",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"}
            },
            "required": ["a", "b"]
        }
    }
}]

# ===============================================================
# Actual multiply function (executed locally)
# ===============================================================
def multiply(a: float, b: float) -> float:
    return a * b

# ===============================================================
# Chat function that handles conversation + tool use
# ===============================================================
def chat_with_bot(user_input, history):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that can also multiply numbers when needed."},
        {"role": "developer", "content": "Only use the multiply tool when a user explicitly or implicitly asks for multiplication."}
    ]

    # Convert history from dicts (Gradio type="messages") to OpenAI messages
    for msg in history:
        if msg["role"] == "user":
            messages.append({"role": "user", "content": msg["content"]})
        elif msg["role"] == "assistant":
            messages.append({"role": "assistant", "content": msg["content"]})

    # Add current user query
    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message
    tool_calls = message.tool_calls

# ===============================================================
    # Case 1: Model requested the multiply tool
# ===============================================================
    if tool_calls:
        print("✅ Tool called:", tool_calls[0].function.name)  # prints multiply
        tool_call = tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = multiply(args["a"], args["b"])

        followup_messages = messages + [
            {
                "role": "assistant",
                "content": None,
                "tool_calls": [{
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }]
            },
            {"role": "tool", "tool_call_id": tool_call.id, "content": str(result)}
        ]

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=followup_messages
        )
        assistant_reply = final_response.choices[0].message.content
# ===============================================================
     # Case 2: No tool needed, assistant answered directly
# ===============================================================
    else:
        print("❌ No tool used, assistant responded directly")
        assistant_reply = message.content

    return assistant_reply

view = gr.ChatInterface(
    fn=chat_with_bot, 
   # type="messages",
    title="Basic Chatbot that can call a tool to multiply two numbers",
    description="Basic Chatbot that can call a tool to multiply two numbers"
)
view.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >6. Get Weather Tool Call with a **Conversational Agent (Gradio app)** with Chat Completion API using OpenAI's gpt-4o-mini </span>

In [ ]:
import requests
import os
from dotenv import load_dotenv
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv('.env', override=True) 
weather_api_key = os.getenv('WEATHER_API_KEY')
client = OpenAI() 

# ===================================================================
# Define the weather tool/function schema
# ===================================================================
tools = [{
    "type": "function",
    "function": {
        "name": "get_city_weather",
        "description": "Get the current weather information for a city. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
        "parameters": {
            "type": "object",
            "properties": {  # "properties" defines the allowed and expected fields inside the object. Each property has its own JSON schema (Here we have one parameter "city_name")
                "city_name": {
                    "type": "string",
                    "description": "The city that the customer wants to know the weather for",
                },
            },
            "required": ["city_name"],
            "additionalProperties": False
        }
    }
}]


# ===================================================================
# Define the function that calls the weather API
# The Tool/Function `get_city_weather("lahore")` will call the weather condition from http://weatherapi.com
# ===================================================================
def get_city_weather(city_name):
    """
    Function to retrieve weather information for a given city.
    Fetches data from WeatherAPI and returns formatted weather information.
    Args:
        city_name (str): The city name to get weather for
    Returns:
        str: Weather information or error message if city not found
    """
    print(f"Tool get_city_weather called for {city_name}")
    city = city_name.strip() # Strip whitespace for consistent processing
    url = f'http://api.weatherapi.com/v1/current.json?key={weather_api_key}&q={city}' # Build API URL
    try:
        response = requests.get(url) # Make API request
        response.raise_for_status()  # Raise error for bad responses (4xx, 5xx)
        weather_data = response.json()
        
        # Extract relevant weather information
        current = weather_data['current']
        location = weather_data['location']
        weather_info = f"Current weather in {location['name']} (Lat: {location['lat']}, Lon: {location['lon']}) as of {location['localtime']}: {current['temp_c']}°C, {current['condition']['text']}"
        return weather_info
        
    except requests.RequestException as e:
        print(f"❌ Error fetching weather data: {e}")
        return "Weather information not available for this city"
    except KeyError as e:
        print(f"Error parsing weather data: {e}")
        return "Weather information not available for this city"




# ==================================================================================================
# Define the function that makes the first call to the model which will either make the tool call or not
#    - It is passed the user’s message and past conversation to the LLM.
#    - It makes the first call to the LLM
#    - If the response `.finish_reason` attribute contains "tool_calls", that means it wants to call a tool.
#    - Then it will call another function `handle_tool_call()` to handle the tool call.
#    - After using the tool, it asks the model again but this time along with the response of the tool
# ===================================================================================================
# System message to define the assistant's behavior and role
system_message = "You are a helpful assistant for a Weather Information service called WeatherAI. "
system_message += "Give your response that should cover the latitude+longitude of the city, its temperature, and other weather related information along with the current date and time in a well formatted fashion."
system_message += "On every request, make a fresh api call to get the latest weather."
system_message += "Always be accurate. If you don't know the answer, say so."


def chat(message, history):
    """
    Main chat function that handles conversation flow with tool calling support.
    Args:
        message (str): Current user message
        history (list): List of previous messages in the conversation
    Returns:
        str: Assistant's response as plain text
    """
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]# Combine the system message, past chat history, and the current user message
    response = client.chat.completions.create(      # Send the conversation to OpenAI with available tools
        model='gpt-4o-mini',  
        messages=messages,             # Complete conversation history
        tools=tools                    # Available tools for the model
        )
        
    # Check if the model's response says: "I want to use a tool"
    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message # Get the model's message containing the tool call request
        tool_response, city = handle_tool_call(assistant_message) # Execute the requested tool
        messages.append(assistant_message)  # Append the Assistant's tool call request to conversation history
        messages.append(tool_response)      # Append the Tool execution result to conversation history
        
        # Send updated conversation back to model for final response
        final_response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages
            )
        return final_response.choices[0].message.content
    else:
        return response.choices[0].message.content # If no tool was called, return the model's direct response

# ==========================
#  User-Defined `handle_tool_call()` function, that executes the appropriate tool/function when the model asks
#      - If the first call to the model detects that it needs to make a call to the tool, it will detect the appropriate function/tool to run
#      - In this example, there is just one tool
#      - Finds the city name the user asked for.
#      - Calls the user-defined `get_city_weather()`.
#      - Sends the result back in a format the model understands.
# ==========================
def handle_tool_call(message):
    """
    Handle tool calls from the model by executing the requested function.
    Args:
        message: OpenAI message object containing tool calls
    Returns:
        tuple: (response_dict, city_name) where response_dict is formatted for OpenAI
    """
    # Extract the first tool call from the model's message
    tool_call = message.tool_calls[0]
    
    # Parse the function arguments from JSON string to Python dictionary
    arguments = json.loads(tool_call.function.arguments)
    
    # Extract the city name from the arguments
    city = arguments.get('city_name')
    
    # Call the tool (Python function) with the extracted city name
    weather_info = get_city_weather(city)
    # Create a properly formatted response for OpenAI's API
    # Return the weather_info directly as content instead of wrapping in JSON
    response = {
        "role": "tool",  # Indicates this is a tool response
        "content": weather_info,  # Return the formatted weather string directly
        "tool_call_id": tool_call.id  # Link response back to the original tool call
    }  
    return response, city
# =================================
# Make Gradio call the chat function
# =================================
interface = gr.ChatInterface(
    fn=chat, 
    #type="messages",
    title="Weather  Forecast Chatbot",
    description="You can give the city name and I will give you current weather information of that city!"
)
interface.launch()

# <span style='background :lightgreen' >7. Executing Shell commands on Local Box with a **Conversational Agent (Gradio app)** with Chat Completion API using OpenAI's gpt-4o-mini </span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">A Chatbot, where other than general chat, a user can ask the LLM to execute different shell commands in natural Language.</h3>

### a. Security Measures
- Command validation: Blocks dangerous commands like rm -rf /, sudo, etc.
- Timeout protection: Commands timeout after 30 seconds
- Error handling: Comprehensive exception handling
- Directory restoration: Always returns to original directory
### b. Cross-Platform Considerations
- macOS: Uses shell=True with standard Unix commands
- Windows: Comments show where to modify for Windows (cmd /c, different commands)
- Path handling: Proper directory management
### c. File Management Capabilities
- The function can handle commands like:
    - ls -la (list files with details)
    - mkdir new_folder (create directories)
    - cp source dest (copy files)
    - find . -name "*.txt" (find files)
    - du -sh * (disk usage)
    - mv old_name new_name (move/rename files)
### d. Example Usage Scenarios:
- "List all files in my current directory"
- "Create a backup folder and copy all my Python files there"
- "Find all large files over 100MB in my home directory"
- "Show me the disk usage of each folder"
- "What's the current date and system uptime?"

### e. Windows modification needed
```
result = subprocess.run(
    ['cmd', '/c', command],  # Use cmd /c for Windows
    capture_output=True,
    text=True,
    timeout=30
)

# And translate commands:
# ls → dir
# du → dir /s
# find → where or dir /s /b
```

##  Step 1: Define Tool Function(s) - These are the actual functions that the LLM can call
- **Here we are having multiple functions, that will execute a given shell command, create a file with content, list some directory contents and so on.**

In [10]:
def execute_shell_command(command):
    """
    Execute a shell command and return the result.
    This function allows the LLM to run any shell command on the system.
    Args:
        command (str): The shell command to execute
    Returns:
        dict: Result containing success status, stdout, stderr, and return code
    """
    try:
        # Run the command in shell, capture both stdout and stderr
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        
        # Return structured result with all relevant information
        return {
            "success": True,
            "stdout": result.stdout,
            "stderr": result.stderr,
            "return_code": result.returncode
        }
    except Exception as e:
        # Return error information if command execution fails
        return {
            "success": False,
            "error": str(e)
        }

def create_file_with_content(filename, content):
    """
    Create a file with specified content using echo command.
    This function allows the LLM to create files with specific content.
    
    Args:
        filename (str): Name of the file to create
        content (str): Content to write to the file
        
    Returns:
        dict: Result of the file creation command
    """
    # Escape double quotes in content to prevent shell injection
    escaped_content = content.replace('"', '\\"')
    
    # Use echo command to write content to file
    command = f'echo "{escaped_content}" > {filename}'
    
    # Execute the command using our shell execution function
    return execute_shell_command(command)

def list_directory(path="."):
    """
    List contents of a directory.
    If no path is provided, list contents of current working directory.
    
    Args:
        path (str): Directory path to list (defaults to current directory)
        
    Returns:
        dict: Result of the directory listing command
    """
    # Use ls -la command to get detailed directory listing
    command = f"ls -la {path}"
    return execute_shell_command(command)

def get_current_directory():
    """
    Get current working directory.
    This helps the LLM understand where it is in the file system.
    
    Returns:
        dict: Result containing current directory path
    """
    command = "pwd"
    return execute_shell_command(command)

def list_executable_files_in_cwd():
    """
    List all executable files in the current working directory.
    This function uses Python's os module instead of shell commands.
    
    Returns:
        dict: Result containing list of executable files in current directory
    """
    try:
        # Get current working directory
        current_dir = os.getcwd()
        
        # List executable files in current directory
        executables = []
        for item in os.listdir(current_dir):
            item_path = os.path.join(current_dir, item)
            
            # Check if item is a file and has execute permissions
            if os.path.isfile(item_path) and os.access(item_path, os.X_OK):
                executables.append(item)
        
        # Return list of executable files
        return {
            "success": True,
            "current_directory": current_dir,
            "executable_files": executables,
            "count": len(executables)
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

def list_executable_files(path):
    """
    List all executable files in a given directory path.
    This function uses Python's os module instead of shell commands.
    
    Args:
        path (str): Directory path to search for executable files
        
    Returns:
        dict: Result containing list of executable files in specified directory
    """
    try:
        # Check if path exists
        if not os.path.exists(path):
            return {"success": False, "error": f"Path '{path}' does not exist"}
        
        # List executable files in specified directory
        executables = []
        for item in os.listdir(path):
            item_path = os.path.join(path, item)
            
            # Check if item is a file and has execute permissions
            if os.path.isfile(item_path) and os.access(item_path, os.X_OK):
                executables.append(item)
        
        # Return list of executable files
        return {
            "success": True,
            "path": path,
            "executable_files": executables,
            "count": len(executables)
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

##  Step 2: Function Schema(s) - JSON schemas that tell the LLM what function(s) are available
- **Since we want our LLM to call a real Python function, so we need to describe all our functions clearly.**
- **To do this, we need to create a list of dictionaries with the name of `tools`, since in this example we have multiple functions, so there are multiple dictionaries in the `tools` list.**
- **Each dictionaryis like a set of instructions or a cheat sheet for the model, to make it understand when and how to call that function during a conversation.**

In [11]:
# ============================================================================
# FUNCTION SCHEMAS - JSON schemas that tell the LLM what functions are available
# =============================================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_shell_command",
            "description": "Execute any shell command on the system",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "The shell command to execute"
                    }
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_file_with_content",
            "description": "Create a file with specified content",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "Name of the file to create"
                    },
                    "content": {
                        "type": "string",
                        "description": "Content to write to the file"
                    }
                },
                "required": ["filename", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": "List the contents of a directory",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Path to the directory (default is current directory)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_directory",
            "description": "Get the current working directory",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_executable_files_in_cwd",
            "description": "List all executable files in the current working directory",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_executable_files",
            "description": "List all executable files in a given directory path",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The directory path to find executable files in"
                    }
                },
                "required": ["path"]
            }
        }
    }
]

In [12]:
# ==========================================
# FUNCTION MAPPING
# Dictionary mapping function names to actual Python functions
# This allows us to dynamically call functions based on the LLM's requests
# ==========================================
available_functions = {
    "execute_shell_command": execute_shell_command,
    "create_file_with_content": create_file_with_content,
    "list_directory": list_directory,
    "get_current_directory": get_current_directory,
    "list_executable_files_in_cwd": list_executable_files_in_cwd,
    "list_executable_files": list_executable_files
}

##  Step 3: User-Defined `chat()` function, let the user chat with the LLM (and tools)
- This function is the main chat engine.
    - It is passed the user’s message and past conversation to the LLM.
    - It makes the first call to the LLM
    - If the response `.finish_reason` attribute contains "tool_calls", that means it wants to call a tool.
    - Then it will call another function `handle_tool_call()` to handle the tool call.
    - After using the tool, it asks the model again but this time along with the response of the tool

In [13]:
# System message to define the assistant's behavior and capabilities
system_message = """You are a helpful assistant that can execute shell commands and perform file operations. 
When a user asks you to list executable files in the 'present working directory', 'current directory', or 'current folder', use the 'list_executable_files_in_cwd' function. 
For other operations:
- List executable files in a specific path: use 'list_executable_files' function
- List directory contents: use 'list_directory' function
- Get current directory: use 'get_current_directory' function
- Create files: use 'create_file_with_content' function
- Execute other commands: use 'execute_shell_command' function
Always complete the full task requested by the user."""

def chat(message, history):
    """
    Main chat function that handles conversation flow with multiple tool calling support.
    
    This is the main function that processes user input and coordinates with OpenAI.
    It handles the complete conversation flow including tool calling.
    
    Args:
        message (str): Current user message (what the user just typed)
        history (list): List of previous messages in the conversation
        
    Returns:
        str: Assistant's response as plain text (what user sees)
    """
    # Build the complete conversation history for OpenAI
    # Structure: [system_message, old_messages..., current_user_message]
    # The system message tells the AI how to behave
    # History contains previous conversation turns
    # Current message is what the user just asked
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    try:
        # === STEP 1: Send initial request to OpenAI ===
        # Send the conversation to OpenAI with available tools
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
            messages=messages,    # Complete conversation history
            tools=tools          # List of available tools the AI can use
        )
        
        # Extract the AI's response message from the API response
        # This contains either text response or tool call requests
        assistant_message = response.choices[0].message
        
        # Add the assistant's message to our conversation history
        # This is important for maintaining context in multi-turn conversations
        messages.append(assistant_message)
        
        # === STEP 2: Check if AI wants to use tools ===
        # The AI can either respond with text OR request to use tools
        # tool_calls will be populated if AI wants to use functions
        if assistant_message.tool_calls:
            
            # Print debug information showing how many tools the AI wants to use
            print(f"\n Number of tool calls requested: {len(assistant_message.tool_calls)}")
            
            # === STEP 3: Execute the requested tools ===
            # Call our helper function to execute all requested tools
            # This runs the actual Python functions and formats results
            tool_responses = handle_tool_calls(assistant_message)
            
            # === STEP 4: Add tool results to conversation ===
            # Add all tool execution results to the conversation history
            # Now the AI will know what the tools returned
            messages.extend(tool_responses)
            
            # === STEP 5: Get final response from AI ===
            # Send the updated conversation (now including tool results) back to OpenAI
            # The AI will use the tool results to generate a human-readable response
            final_response = openai_client.chat.completions.create(
                model="gpt-4o-mini",  # llama-3.1-8b-instant, openai/gpt-oss-20b, openai/gpt-oss-120b, qwen/qwen3-32b, meta-llama/llama-4-scout-17b-16e-instruct, meta-llama/llama-4-maverick-17b-128e-instruct
                messages=messages  # This now includes tool results
            )
            
            # Return the AI's final response after using the tools
            # This is what the user will see - a natural language response
            return final_response.choices[0].message.content
        
        else:
            # === ALTERNATIVE: No tools needed ===
            # If the AI didn't request any tools, return its direct response
            # This happens for simple questions that don't need function calls
            print("\n No tool calls requested")
            return assistant_message.content
            
    except Exception as e:
        # === ERROR HANDLING ===
        # If anything goes wrong during the process, catch the error
        # Print the error for debugging and return user-friendly message
        print(f"Error in chat function: {e}")
        return "I'm sorry, I encountered an error while processing your request."


##  Step 4: User-Defined `handle_tool_call()` function, that executes the appropriate tool/function when the model asks
- If the first call to the model detects that it needs to make a call to the tool, it will detect the appropriate function/tool to run
- In this example, there is multiple functions, it decides the function to call, extract the required parameters, and execute it on your local box. 
- Finally, it sends the result back in a format the model understands, as context.

In [14]:
def handle_tool_calls(message):
    """
    Handle multiple tool calls from the model by executing the requested functions.
    
    This function processes the model's requests to use tools/functions.
    When the model decides it needs to call a function (like executing a shell command),
    this function executes those functions and formats the results.
    
    Args:
        message: message object containing tool calls
        
    Returns:
        list: List of tool response dictionaries formatted for model
    """

    # Initialize empty list to store all tool execution results
    # Each tool call will generate one response that goes in this list
    tool_responses = []
    
    # Process each tool call requested by the model
    # Loop through each tool call that the model requested
    # The model can request multiple tools in a single response
    # For example: "list directory AND create a file"
    for tool_call in message.tool_calls:
        
        # Extract the name of the function the model wants to call
        # Example: "execute_shell_command" or "create_file_with_content"
        function_name = tool_call.function.name
        
        # Extract the arguments the model wants to pass to the function
        # Arguments come as JSON string, so we parse it into Python dictionary
        # Example: '{"command": "ls -la"}' becomes {"command": "ls -la"}
        function_args = json.loads(tool_call.function.arguments)
        
        # Print debug information to console (helpful for development)
        # Shows which function is being called and what arguments are being passed
        print(f"\n🔧 Executing: {function_name}")
        print(f"📝 Arguments: {function_args}")
        
        # Check if the requested function exists in our available_functions dictionary
        # This prevents errors if the model tries to call a non-existent function
        if function_name in available_functions:
            
            # Call the actual Python function with the provided arguments
            # The **function_args unpacks the dictionary as keyword arguments
            # Example: execute_shell_command(**{"command": "ls -la"}) 
            # becomes: execute_shell_command(command="ls -la")
            function_result = available_functions[function_name](**function_args)
            
            # Create a properly formatted response dictionary for OpenAI's API
            # This tells OpenAI what the function returned
            tool_response = {
                "tool_call_id": tool_call.id,  # Unique ID linking response to the original request
                "role": "tool",                # Indicates this message is from a tool execution
                "name": function_name,         # Name of the function that was executed
                "content": json.dumps(function_result)  # Function result as JSON string
            }
            
            # Add this tool response to our list of responses
            tool_responses.append(tool_response)
            
        else:
            # Handle the case where the model requested a function that doesn't exist
            # Create an error response instead of crashing the program
            error_response = {
                "tool_call_id": tool_call.id,  # Same ID as the original request
                "role": "tool",                # Still a tool response, but with error
                "name": function_name,         # The function name that wasn't found
                "content": json.dumps({"success": False, "error": f"Function {function_name} not found"})
            }
            # Add the error response to our list
            tool_responses.append(error_response)
    
    # Return all tool responses to be added to the conversation
    return tool_responses



##  Step 5: Use Gradio and call the `chat()` Function for Testing your Code
- Ask the following Questions:
    -  Hello there, which is the capital of Pakistan
    -  Tell me a joke
    -  Please create a file in the present working directory with the name of abc.txt and write text in it saying "Welcome all to the world of tools in LLM".
    -  Please tell me the absolute path of the present working directory
    -  Please list the contents of directory "/Users/"
    -  Please execute the shell command "ps"
    -  Please execute the shell command "ls"
    -  Please list all executable files in the present working directory
    -  Can you tell me the ip address of my local machine
    -  Please run the ifconfig command
    -  Can you run the ls command in the present working directory, and list the executable files in it along with the count

In [15]:
import os                                   # Provides access to environment variables and operating system utilities.
import subprocess
import pytz
import rich                                 # Renders richly formatted output (colors, tables, tracebacks) in the terminal.
import json                                 # Encodes and decodes data in JSON format.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.
from groq import Groq                        # Official Groq Python client for interacting with Groq-hosted LLMs via their API.

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY') 
groq_api_key = os.getenv("GROQ_API_KEY")
openai_client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)  # This client is your gateway to native OpenAI models like gpt-4o, gpt-4o-mini, etc.
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.




import gradio as gr
interface = gr.ChatInterface(
    fn=chat, 
    #type="messages",
    title="Shell Command Assistant",
    description="""
    I can help you execute shell commands, create files, and perform file operations.You ay try asking the following Questions:\n
    Please tell me the absolute path of the present working directory\n
    Please list the contents of "/Users/" directory.\n
    Please execute the shell command "ps"\n
    Can you tell me the ip address of my local machine which is running Mac OS\n
    Please create a file in the present working directory with the name of abc.txt and write text in it saying "Welcome all to the world of tools in LLM\n
    Can you run the ls command in the present working directory, and list only the files with .mp3 extension
"""
)
interface.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
